# 02 — Exploratory Data Analysis

Every plot is generated by `src/eda.py` and saved to `reports/images/` (same figures the README
uses). Below each figure: what it shows and why it matters for modeling.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import Image, display
from src.utils import load_config, resolve_path

config = load_config()
pd.set_option("display.max_columns", 40)

In [ ]:
from src.data_loader import load_raw_data
from src import eda

df = load_raw_data(config)
paths = eda.run_eda(df, config)   # regenerates all figures
[p.name for p in paths]

## Class distribution — the defining property

Genuine outnumbers fraud ~578:1. Every modeling decision downstream (metric choice, resampling,
threshold tuning) exists because of this imbalance. Note the log scale — on a linear scale the fraud
bar is invisible.

In [ ]:
display(Image(str(resolve_path(config["paths"]["images_dir"]) / "class_distribution.png")))

## Transaction amounts

Fraud amounts skew **small** (median ≈ $9 vs $22 genuine) — fraudsters test cards with small charges
— but the fraud distribution is heavy-tailed, so rare large frauds dominate dollar losses. This is
why our business-cost threshold weights each missed fraud by its actual amount. The raw distribution
is extremely right-skewed, motivating the `log1p(Amount)` feature.

In [ ]:
display(Image(str(resolve_path(config["paths"]["images_dir"]) / "amount_distribution.png")))

## Time of day

Fraud **rate** spikes in the early-morning hours (roughly 2–5 AM): cardholders are asleep, so
stolen-card activity faces no competition from genuine spending and less chance of immediate
detection. This pattern is the rationale for the engineered `Hour` feature.

In [ ]:
display(Image(str(resolve_path(config["paths"]["images_dir"]) / "time_distribution.png")))

## Correlations

`V1–V28` are PCA components, so they are mutually orthogonal (the off-diagonal V-block is ~0).
The interesting row is `Class`: several components (V17, V14, V12, V10…) correlate noticeably with
fraud — they will dominate feature importance later. Multicollinearity is a non-issue here.

In [ ]:
display(Image(str(resolve_path(config["paths"]["images_dir"]) / "correlation_heatmap.png")))

## Fraud vs genuine — the most discriminative features

Kernel density estimates for the V-features most correlated with the target. For V14/V12/V10 the
fraud distribution is shifted far left of genuine — a clean, learnable signal, which is why even a
linear model reaches ROC-AUC ≈ 0.98 on this dataset.

In [ ]:
display(Image(str(resolve_path(config["paths"]["images_dir"]) / "fraud_vs_genuine_features.png")))

## Outliers

Genuine amounts reach $25,691 while the largest fraud is $2,126. We deliberately do **not** clip or
remove outliers: extreme-but-genuine amounts are real behavior the model must tolerate, and
tree ensembles are robust to them. `log1p` + `StandardScaler` handle them for the linear model.

In [ ]:
display(Image(str(resolve_path(config["paths"]["images_dir"]) / "outlier_analysis.png")))